# C2: Detect Indonesian PII in prompts (English)
**Platform:** NeMo Guardrails (GA, RHOAI 3.5) with a custom action for Indonesian identifiers (NIK, NPWP, account number, +62 phone, DOB, mother's maiden name) and Presidio names/e-mail.
**Config:** `policy-pii-detect` (block on any PII). **Datasets:** synthetic DS1-EN (180 positives / 60 hard negatives) and a hand-written **held-out** set (37 / 12) that was never used for tuning. Report the held-out numbers.

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
def show(rows, cols=("group","n","tp","fn","fp","tn","recall","fpr","p50_latency_s")):
    import pandas as pd
    df = pd.DataFrame(rows)[[c for c in cols if c in rows[0]]]
    display(df.style.format(precision=3, na_rep=""))
    return df

**How to read the results table** (one row per group, plus `overall`):

| Column | Meaning |
|---|---|
| `n` | prompts in this group |
| `tp` | harmful prompts the guardrail **blocked** (correct) |
| `fn` | harmful prompts that got **through** (misses) |
| `fp` | harmless prompts that were **wrongly blocked** (false alarms) |
| `tn` | harmless prompts that **passed** (correct) |
| `recall` | `tp / (tp + fn)`: share of harmful prompts caught. 1.000 = all caught |
| `fpr` | `fp / (fp + tn)`: share of harmless prompts wrongly blocked. 0.000 = none |
| `p50_latency_s` | median time for one guardrail check, in seconds |

Groups that contain only harmful prompts have no `fpr`; groups of only harmless prompts have no `recall` (shown blank).

Here a "harmful" prompt is one that contains personal data, and **blocked** means the guardrail detected it. Groups are PII types (`nik`, `npwp`, `account_number`, `phone`, `email`, `name`, `address`, `dob`, `mother_maiden_name`); `hard_negative` holds normal banking prompts that look a bit like PII (street names, tracking numbers) and must **not** be blocked.

## Held-out set (the honest number)

In [ ]:
ho = h.evaluate("data/ds1_holdout_en.jsonl", mode="check", config_id="policy-pii-detect", group_field="category", workers=6, errors_as="exclude")
df_ho = show(h.metrics(ho, group_field="category"))
[ (x["category"], x["text"]) for x in ho if x["correct"] is False ]

**What this shows:** the held-out set (37 prompts with PII, 12 without) was written separately and never used while building the rules, so it's the fair measure.

* `overall` → `recall` **1.000** means every one of the 37 PII prompts was caught.
* `hard_negative` → `fp` **2** of 12 means two normal prompts were wrongly blocked (`fpr` ≈ 0.167).
* The list printed under the table is exactly those wrongly blocked prompts: a street name ("Jalan raya") read as an address, and a courier tracking number read as an account number. That's the trade-off of catching every real identifier.

If your numbers differ slightly, the model-based parts (names) can vary run to run; structured IDs (NIK, NPWP, phone…) are rule-based and should not change.

## DS1-EN (development set, used for tuning)

In [ ]:
ds1 = h.evaluate("data/ds1_pii_en.jsonl", mode="check", config_id="policy-pii-detect", group_field="category", workers=6, errors_as="exclude")
df = show(h.metrics(ds1, group_field="category"))
ax = df[df.group.isin([g for g in df.group if g not in ("overall","hard_negative")])].plot.bar(x="group", y="recall", legend=False, ylim=(0,1.05), title="Recall per PII type (DS1-EN)")

**What this shows:** the development set (180 PII prompts, 60 without) was used while tuning the rules, so expect near-perfect results: `recall` 1.000 and `fp` 0. That's why the held-out numbers above are the ones to quote.

The bar chart shows recall per PII type; every bar at 1.0 means every type was detected in every prompt.

## One example, end to end

In [ ]:
r = h.check("My NIK is 3171041208850123, please update my KYC.", config_id="policy-pii-detect")
print(r["blocked"], json.dumps(r["raw"]["rails_status"]))

**Reading the output:** `True` means the prompt was blocked. The JSON names the rail that fired, here the custom action `detect indonesian identifiers on input`, which recognised the 16-digit NIK. A normal question would print `False` and an empty or `success` status.